# Overhead điều phối Langflow với flow SCRFD

Notebook đọc kết quả **offline mặc định**. Chỉ khi chủ động đặt `RUN_LIVE = True` mới đóng băng fixture, khởi động worker và gửi request. Dữ liệu được đọc từ raw của `RUN_DIR`; không có số liệu tốc độ giả lập.

Campaign factorial có 1.000 request đo/arm **00 / 01 / 10 / 11**, bốn block cân bằng. Bit đầu là compilation cache, bit sau là warm graph registry. Warmup tách khỏi 4.000 mẫu đo. Profile v3 thêm fixture DB/storage độc lập, RAM checkpoints và provenance; primary campaign giữ diagnostic phases off.

`RUN_DIR` mặc định là `main-v2` lịch sử: latency vẫn phân tích được, **RAM chưa đo**. Không gộp main/main-v2/v3 hoặc các child diagnostic thành một dataset. `requests.Session` có thể tái sử dụng kết nối TCP; cờ HTTP-reuse legacy tắt không có nghĩa mọi connection reuse đều tắt.


## Ranh giới đo và phạm vi

`langflow_overhead_ms = server_total_ms - scrfd_processing_ms`

- `server_total_ms`: wall-clock monotonic trong worker từ điểm vào ASGI ngoài cùng của ứng dụng cho `/run` đến khi gửi xong response body.
- `scrfd_processing_ms`: độ dài **hợp** các khoảng output method của bốn node SCRFD, gồm preprocessing, inference, vẽ/lưu ảnh; phần chồng lấn chỉ trừ một lần.
- Chat Input/Output, chuẩn bị graph/component, validation, auth, middleware và serialize response vẫn thuộc overhead. Không trừ thêm GC, DB, telemetry hay estimated observer cost.
- Upload, tải ảnh kết quả và mạng client nằm ngoài chỉ số chính. Đây không phải CPU time hoặc latency end-user.

API production vẫn dùng `input_type=chat`, `output_type=chat`, `stream=false`, kiểm tra số mặt và hash pixels ảnh so với reference. Compilation cache lưu artifact biên dịch source; benchmark không thêm cache component instance, ONNX session hay kết quả inference.

Đợt v3 bổ sung kiểm soát phép đo và bằng chứng, không tuyên bố đã tối ưu production runtime. Phạm vi RAM là **toàn serving worker**, một flow/concurrency 1; bốn checkpoints không đo peak RAM. Payload/cache accounting là bytes dữ liệu đang giữ, không phải heap/RSS attribution.


In [ ]:
import html
import json
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, Image, display

_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / "benchmark_analyst" / "reporting.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Mở notebook từ repository langflow_CV hoặc đặt REPO_ROOT tuyệt đối.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUNS = REPO_ROOT / "benchmark_analyst" / "runs"
RUN_DIR = RUNS / "main-v2"  # Dataset lịch sử mặc định, chỉ phân tích offline.
RUN_LIVE = False
RUN_REPEAT_LIVE = False
RUN_DIAGNOSTIC_LIVE = False
SUITE_DIR = None  # Đặt Path tới suite diagnostic có sẵn để phân tích offline.

LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
INPUT_CONFIG = RUNS / "reference" / "config.json"  # Reference pixels đã xác minh.
ENV_FILE = REPO_ROOT / ".env"
CREDENTIALS_FILE = RUNS / "inputs" / "credentials.env"
FIXTURE_DIR = RUNS / f"fixture-v3-{LABEL}"
SMOKE_DIR = RUNS / f"smoke-v3-{LABEL}"
MAIN_DIR = RUNS / f"main-v3-{LABEL}"
REPEAT_DIR = RUNS / f"main-v3-repeat-{LABEL}"
DIAGNOSTIC_SMOKE_DIR = RUNS / f"diagnostic-smoke-v3-{LABEL}"
DIAGNOSTIC_DIR = RUNS / f"diagnostic-v3-{LABEL}"
SOURCE_DATABASE = None  # Override Path nếu .env không xác định được local SQLite.
SOURCE_STORAGE = None  # Override Path tới storage cùng DB khi cần.
PORT = 7867
REQUESTS_PER_ARM = 1000
BLOCKS = 4
WARMUPS = 5
print("Chế độ:", "LIVE" if RUN_LIVE else "OFFLINE", "| Dataset:", RUN_DIR)


## Thiết kế và dữ liệu đầu vào

| Block | Thứ tự arm |
| --- | --- |
| 1 | 00 → 01 → 11 → 10 |
| 2 | 01 → 10 → 00 → 11 |
| 3 | 10 → 11 → 01 → 00 |
| 4 | 11 → 00 → 10 → 01 |

Mỗi slot clone cùng baseline DB/storage rồi khởi động một worker mới: **5 warmup + 250 request đo**. Tổng 16 slot, 4.000 mẫu đo, 80 warmup và 64 RAM records/campaign. Repeat là campaign riêng, không ghép mẫu giữa hai run.

`INPUT_CONFIG` chứa `flow_id`, `input_node_id`, `output_node_id`, bốn `scrfd_node_ids`, `image_path`, `model_path`, `flow_export_path`, `expected_faces`, `timeout_seconds` và `reference_pixel_sha256`. Đường dẫn tương đối tính từ config. API key nằm trong credentials dotenv hoặc môi trường; không chép key vào notebook.

Chuẩn bị workspace theo `uv.lock` và tools benchmark đã pin bằng `uv pip install -r benchmark_analyst/requirements.txt` trước khi bật live. CLI preflight ghi actual versions và kiểm tra dependency; `--no-sync` chỉ dùng sau bước chuẩn bị này. Reference có sẵn được giữ nguyên; nếu chưa có reference, làm theo bước prepare trong README trước khi freeze.


In [ ]:
CLI = ["uv", "run", "--no-sync", "python", "-m", "benchmark_analyst.benchmark_langflow"]
AUTH_ARGS = ["--env-file", str(ENV_FILE), "--credentials-file", str(CREDENTIALS_FILE), "--port", str(PORT)]
CONTROLLED_ARGS = ["--fixture", str(FIXTURE_DIR), "--memory-checkpoints"]


def run_cli(arguments):
    command = CLI + [str(arg) for arg in arguments]
    print(shlex.join(command))  # Hiển thị đường dẫn, không đọc/hiển thị credentials.
    return subprocess.run(command, cwd=REPO_ROOT, check=True)


## Tùy chọn chạy thật: đóng băng fixture

Cell chỉ thực thi khi `RUN_LIVE=True`. `freeze` tạo consistent SQLite backup và storage snapshot dưới `runs/`; nó không chạy reference request. DB/storage đang dùng của người dùng là nguồn đọc; các worker tiếp theo chỉ ghi bản clone riêng mỗi slot. Nếu backend không phải local SQLite/filesystem hoặc không chứng minh snapshot nhất quán, preflight phải dừng.

Giữ source, config, model, input, reference và fixture cố định từ trước smoke đến hết main/repeat/suite. Mỗi output là thư mục mới; không sửa notebook/source trong lúc đo.


In [ ]:
if RUN_LIVE:
    frozen_config = json.loads(INPUT_CONFIG.read_text(encoding="utf-8"))
    if not frozen_config.get("reference_pixel_sha256"):
        raise RuntimeError("INPUT_CONFIG phải trỏ reference/config.json đã có frozen pixel hash.")
    freeze_args = ["freeze", "--config", INPUT_CONFIG, "--output", FIXTURE_DIR, *AUTH_ARGS]
    if SOURCE_DATABASE is not None:
        freeze_args.extend(["--source-database", SOURCE_DATABASE])
    if SOURCE_STORAGE is not None:
        freeze_args.extend(["--source-storage", SOURCE_STORAGE])
    run_cli(freeze_args)
else:
    print("OFFLINE: bỏ qua freeze; không tạo worker hoặc gửi request.")


## Tùy chọn chạy thật: smoke v3 với RAM

Smoke có 8 request/arm, hai request/slot và 5 warmup. Cùng fixture/profile/source với primary main, RAM on, diagnostic phases off. Smoke chỉ xác minh đường chạy. Gate dùng **overall validity**, gồm latency/cache/output và mandatory RSS; USS bị từ chối có reason không làm RSS invalid.


In [ ]:
if RUN_LIVE:
    run_cli(["run", "--config", INPUT_CONFIG, "--output", SMOKE_DIR,
             "--requests-per-arm", 8, "--blocks", BLOCKS, "--warmups", WARMUPS,
             *CONTROLLED_ARGS, *AUTH_ARGS])
    smoke_analysis = json.loads((SMOKE_DIR / "analysis.json").read_text(encoding="utf-8"))
    if not smoke_analysis.get("overall_valid", False):
        raise RuntimeError("Smoke overall INVALID: xem latency/RAM/diagnostic findings trước main.")
    print("Smoke overall VALID:", SMOKE_DIR / "report.html")
else:
    print("OFFLINE: bỏ qua smoke.")


## Tùy chọn chạy thật: primary main và repeat

`--smoke` phải khớp workload/source, profile, fixture hash, RAM declaration và diagnostic mode. Primary có 1.000 measured request/arm, 4.000 tổng; không có diagnostic drains giữa requests. Không chạy test/build hoặc tác vụ nặng cùng lúc. Giữ raw lỗi, không retry bù hay resume run dở dang.

`RUN_REPEAT_LIVE=True` chạy thêm một campaign riêng cùng profile/source/fixture vào `REPEAT_DIR`. Đối chiếu hướng mean/median/tails giữa hai báo cáo, không ghép paired blocks của hai run. Dữ liệu lịch sử main-v2 giữ nguyên.


In [ ]:
if RUN_LIVE:
    primary_args = ["run", "--config", INPUT_CONFIG, "--smoke", SMOKE_DIR,
                    "--requests-per-arm", REQUESTS_PER_ARM, "--blocks", BLOCKS,
                    "--warmups", WARMUPS, *CONTROLLED_ARGS, *AUTH_ARGS]
    run_cli([*primary_args, "--output", MAIN_DIR])
    RUN_DIR = MAIN_DIR  # Chỉ đổi dataset offline sau khi campaign mới hoàn tất.
    if RUN_REPEAT_LIVE:
        run_cli([*primary_args, "--output", REPEAT_DIR])
        print("Repeat riêng:", REPEAT_DIR / "report.html")
else:
    print("OFFLINE: bỏ qua main/repeat; các cell dưới đọc RUN_DIR.")


## Tùy chọn chạy thật: diagnostic suite D0–D3

Chỉ thực thi khi cả `RUN_LIVE=True` và `RUN_DIAGNOSTIC_LIVE=True`. Chạy diagnostic smoke riêng trước suite dài; primary smoke không chứng thực các diagnostic treatments.

- D0: observer phases off nhưng cùng cadence control/drains của suite.
- D1: coarse observer, GC/event-loop lag; D1−D0 mô tả ảnh hưởng cả observer package.
- D2: như D1 nhưng product telemetry policy off; native tracing vẫn on.
- D3: như D1 nhưng 20 warmup thay 5, bao gồm thêm message/job/trace writes.

Bốn variants interleave theo outer block, mỗi child chỉ có arms 10/11. Suite dài có 1.000 measured request/arm/child = 8.000 tổng, không phải factorial campaign. Timings diagnostic không nhập vào primary latency/RAM contrasts; phase evidence và spike overlap không tự chứng minh nguyên nhân.


In [ ]:
if RUN_LIVE and RUN_DIAGNOSTIC_LIVE:
    suite_args = ["diagnose", "--config", INPUT_CONFIG, "--fixture", FIXTURE_DIR,
                  "--memory-checkpoints", "--blocks", BLOCKS, *AUTH_ARGS]
    run_cli([*suite_args, "--requests-per-arm", 8, "--output", DIAGNOSTIC_SMOKE_DIR])
    smoke_suite = json.loads((DIAGNOSTIC_SMOKE_DIR / "suite_analysis.json").read_text(encoding="utf-8"))
    if not smoke_suite.get("overall_valid", False):
        raise RuntimeError("Diagnostic smoke overall INVALID: xem suite/child findings.")
    run_cli([*suite_args, "--requests-per-arm", 1000, "--smoke", DIAGNOSTIC_SMOKE_DIR,
             "--output", DIAGNOSTIC_DIR])
    SUITE_DIR = DIAGNOSTIC_DIR
else:
    print("Bỏ qua live diagnostic suite; có thể đặt SUITE_DIR để đọc suite có sẵn offline.")


## Phân tích offline: bốn trạng thái riêng

`analyze` đọc `manifest.json`, `requests.jsonl`, `worker_evidence.jsonl` và sidecars đã khai báo. `state.json` nếu có phải COMPLETE. Nó không gọi API hay sửa raw; ghi lại JSON/CSV/Markdown/HTML/charts là derived artifacts.

`valid` tiếp tục chỉ latency/cache/output. `resource_validity` và `diagnostic_validity` kiểm tra evidence riêng; `overall_valid` là AND của latency với các extension bắt buộc đã khai báo. Dataset legacy không khai báo extension giữ `overall_valid=valid`, với RAM/observer **unavailable/not_requested**. Không chuyển RAM chưa đo hay USS denied thành zero.


In [ ]:
from benchmark_analyst.reporting import analyze

analysis = None
if RUN_DIR.is_dir():
    analysis = analyze(RUN_DIR)
    resource = analysis.get("resource_validity", {"requested": False, "valid": True, "status": "unavailable", "reason": "not_requested"})
    diagnostic = analysis.get("diagnostic_validity", {"requested": False, "valid": True, "status": "unavailable", "reason": "not_requested"})
    print("Latency/cache/output:", "VALID" if analysis["valid"] else "INVALID")
    print("RAM:", resource["status"], "| requested:", resource["requested"], "| reason:", resource.get("reason"))
    print("Diagnostic:", diagnostic["status"], "| requested:", diagnostic["requested"], "| reason:", diagnostic.get("reason"))
    print("Overall:", "VALID" if analysis.get("overall_valid", analysis["valid"]) else "INVALID")
    print("Kind/profile:", analysis.get("kind", "factorial"), analysis["manifest"].get("profile_id", "legacy/unavailable"))
    print("Scope:", "FACTORIAL 1.000 request/arm" if analysis["is_full_campaign"] else "SMOKE / diagnostic / cấu hình thử")
    for label, messages in (("Latency", analysis["validation_errors"]), ("RAM", resource.get("errors", [])),
                            ("Diagnostic", diagnostic.get("errors", []))):
        for message in messages:
            print(f"- {label}: {message}")
    if not resource["requested"]:
        print("RAM chưa đo: legacy dataset không có RSS/USS checkpoints; không suy footprint từ cache payload.")
else:
    print("Chưa có thư mục raw. Đặt RUN_DIR tới kết quả có sẵn rồi chạy lại cell offline.")


## Mean, p50, p95, p99 và RAM coverage

Latency statistics không gồm warmup, percentile nội suy tuyến tính type 7; giữ mọi outlier. Đọc mean/median và tails riêng, không chọn winner chung khi chưa có SLA/RAM budget. Method envelope decomposition có pre/gaps/post/chat khi đủ sáu spans tuần tự; decomposition unavailable không tự làm phép union SCRFD invalid.

RAM có absolute RSS/USS ở `before_warmup`, `after_warmup`, `after_measurement`, `after_idle`; n = số worker độc lập/arm, không phải số request. `before_warmup` đã qua readiness/workload verification. `after_idle` bắt đầu ít nhất 5 giây sau sample cuối gồm download/output verification, không khẳng định quiescence.

Các paired differences giữ dấu: arm−00 cùng checkpoint/block, marginal 11−10 và difference-in-differences `(after_idle−before_warmup)_arm − (...)_control`. Raw bytes là integer không âm; MiB = bytes/1.048.576. Signed deltas âm là hợp lệ. Tỷ số ms tiết kiệm/MiB chỉ có nghĩa mô tả khi RAM tăng dương và đủ coverage; không dùng payload bytes làm RAM.


In [ ]:
if analysis is not None:
    display(HTML((RUN_DIR / "report.html").read_text(encoding="utf-8")))
    resource = analysis.get("resource_validity", {})
    if not resource.get("requested", False):
        print("RAM chưa đo / unavailable cho dataset này.")
    elif not resource.get("valid", False):
        print("RAM INVALID: chỉ giữ diagnostics, không kết luận RAM trade-off.")
    else:
        print("Coverage:", json.dumps(resource.get("metrics", {}), ensure_ascii=False))
        print("RAM toàn worker; cache payload accounting được ghi riêng trong report.")
        ram_rows = []
        for arm, checkpoints in resource.get("arms", {}).items():
            for checkpoint, metrics in checkpoints.items():
                rss, uss = metrics["rss"], metrics["uss"]
                rss_mib = "unavailable" if rss["mean"] is None else f"{rss['mean'] / 1048576:.3f}"
                uss_mib = "unavailable" if uss["mean"] is None else f"{uss['mean'] / 1048576:.3f}"
                ram_rows.append([arm, checkpoint, str(rss["n"]), rss_mib, str(uss["n"]), uss_mib])
        columns = ["Arm", "Checkpoint", "n RSS workers", "Mean RSS MiB", "n USS workers", "Mean USS MiB"]
        table = "<table><thead><tr>" + "".join(f"<th>{html.escape(value)}</th>" for value in columns) + "</tr></thead><tbody>"
        table += "".join("<tr>" + "".join(f"<td>{html.escape(value)}</td>" for value in row) + "</tr>" for row in ram_rows)
        display(HTML(table + "</tbody></table>"))


In [ ]:
if analysis is not None:
    for filename, caption in (("charts.png", "Latency percentiles, block means và request sequence"),
                              ("tradeoffs.png", "Latency–RAM: cùng campaign/profile, RAM coverage đã kiểm tra"),
                              ("diagnostics.png", "Method/phase/spike evidence của run này; không nhập diagnostic timings vào primary")):
        image_path = RUN_DIR / filename
        if image_path.is_file():
            print(caption)
            display(Image(filename=str(image_path)))
        else:
            print(filename, "unavailable: chưa có figure/evidence cho dataset này.")


## Diễn giải hiệu ứng và giới hạn

Với μ là mean overhead, Δ âm nghĩa là overhead giảm. Factorial contrasts gồm μ10−μ00, μ01−μ00, μ11−μ00; conditional warm effect là μ11−μ10, interaction là μ11−μ10−μ01+μ00. `reduction_pct` dương nghĩa là overhead giảm; không giả định thêm cache luôn làm mean/median/tails tốt hơn.

Latency `contrasts=None` khi latency invalid; RAM contrasts unavailable/invalid có gate riêng. Một run latency VALID nhưng mandatory RSS thiếu có **overall INVALID**. Không tuyên bố trade-off đã kiểm chứng chỉ từ nhãn latency VALID. Smoke chỉ xác minh đường chạy.

Chỉ bốn block độc lập; 4.000 requests không phải 4.000 independent repetitions của điều kiện máy. Không có CI/p-value hoặc causal claim. Baseline runtime tương lai phải là campaign cùng v3 profile/source/fixture, không lấy main-v2 làm đối chứng before cho runtime fix dưới profile mới.


In [ ]:
if analysis is not None:
    if not analysis.get("overall_valid", analysis["valid"]):
        print("Overall INVALID: chưa kết luận trade-off hoàn chỉnh; xem từng validity gate.")
    elif not analysis["is_full_campaign"]:
        print("Smoke/diagnostic: không phải full factorial campaign.")
    else:
        print("Latency contrasts:", json.dumps(analysis["contrasts"], ensure_ascii=False, indent=2))
        resource = analysis.get("resource_validity", {})
        if resource.get("requested") and resource.get("valid"):
            print("Signed RAM contrasts (bytes):", json.dumps(resource.get("contrasts"), ensure_ascii=False, indent=2))
        else:
            print("RAM chưa đo: giữ kết luận latency ở phạm vi dataset lịch sử.")
    print("Derived artifacts có sẵn:")
    for filename in ("analysis.json", "report.md", "report.html", "summary.csv", "resource_summary.csv", "charts.png", "tradeoffs.png", "diagnostics.png"):
        artifact = RUN_DIR / filename
        if artifact.is_file():
            print(artifact)


## Tùy chọn phân tích diagnostic suite offline

Đặt `SUITE_DIR` tới một suite có sẵn. Analyzer kiểm tra suite/child COMPLETE, lịch 32 slots, source/workload/fixture/profile, RSS và diagnostic coverage; ghi `suite_analysis.json`, `suite_report.md/html` rồi giữ riêng child reports. D0 phases off không cần phase events.

Paired comparisons theo cùng outer block/arm: D1−D0 (observer package), D2−D1 (product telemetry policy), D3−D1 (warmup policy bao gồm thêm DB writes). Phase overlap chỉ là evidence; khi observer thay đổi hiệu ứng cùng cỡ đang giải thích, giữ attribution ở mức giả thuyết. Không trộn suite vào primary statistics.


In [ ]:
suite_analysis = None
if SUITE_DIR is not None:
    from benchmark_analyst.suite_reporting import analyze_suite

    selected_suite = Path(SUITE_DIR)
    if selected_suite.is_dir():
        suite_analysis = analyze_suite(selected_suite)
        print("Suite overall:", "VALID" if suite_analysis["overall_valid"] else "INVALID")
        for message in suite_analysis["errors"]:
            print("-", message)
        for variant, child in suite_analysis["children"].items():
            print(variant, "latency:", child.get("valid"), "RAM:", child.get("resource_validity", {}).get("status"),
                  "diagnostic:", child.get("diagnostic_validity", {}).get("status"), "overall:", child.get("overall_valid"))
        display(HTML((selected_suite / "suite_report.html").read_text(encoding="utf-8")))
        print("Suite artifacts:", selected_suite / "suite_analysis.json", selected_suite / "suite_report.html")
    else:
        print("SUITE_DIR không tồn tại; đặt tới suite raw có sẵn.")
else:
    print("Chưa chọn diagnostic suite; không chạy live hoặc gọi API.")


## Tái lập bằng CLI

Phân tích offline (legacy RAM vẫn unavailable):

```bash
uv run --no-sync python -m benchmark_analyst.benchmark_langflow analyze \
  --output benchmark_analyst/runs/main-v2
uv run --no-sync python -m benchmark_analyst.benchmark_langflow analyze-suite \
  --output benchmark_analyst/runs/diagnostic-v3
```

Ví dụ live với reference đã có, output mới và source đã cố định:

```bash
uv run --no-sync python -m benchmark_analyst.benchmark_langflow freeze \
  --config benchmark_analyst/runs/reference/config.json \
  --credentials-file benchmark_analyst/runs/inputs/credentials.env \
  --output benchmark_analyst/runs/fixture-v3
uv run --no-sync python -m benchmark_analyst.benchmark_langflow run \
  --config benchmark_analyst/runs/reference/config.json \
  --credentials-file benchmark_analyst/runs/inputs/credentials.env \
  --fixture benchmark_analyst/runs/fixture-v3 --memory-checkpoints --port 7867 \
  --requests-per-arm 8 --blocks 4 --warmups 5 \
  --output benchmark_analyst/runs/smoke-v3
uv run --no-sync python -m benchmark_analyst.benchmark_langflow run \
  --config benchmark_analyst/runs/reference/config.json \
  --credentials-file benchmark_analyst/runs/inputs/credentials.env \
  --fixture benchmark_analyst/runs/fixture-v3 --memory-checkpoints --port 7867 \
  --smoke benchmark_analyst/runs/smoke-v3 --requests-per-arm 1000 --blocks 4 --warmups 5 \
  --output benchmark_analyst/runs/main-v3
```

Diagnostic tùy chọn: dùng `diagnose` cùng config/fixture với `--requests-per-arm 8` vào suite smoke mới; suite dài dùng `--requests-per-arm 1000 --smoke <diagnostic-smoke-dir>` và output riêng. Mỗi command phân tích/đo trả 0 khi **overall VALID**, 2 khi invalid. Xem README cho dependency/provenance, source locations và lifecycle của DB/storage clones.
